# Coarse confidence against error (coarse stage only, no LoFTR, no DINO)

**Question.** Above ~800 m the pipeline skips photos whose coarse peak is often right (iteration 24). Is there a confidence that does NOT use the truth and separates right coarse answers from wrong ones, per height band?

**What it measures.** For every photo, the 6 standard queries (headings -5/0/+5 x scales 1.0/1.05) are drawn and scored by the edge-orientation ZNCC over the 5.5 km disc (hint jittered 5 km, as in the real test). Per query we keep the best peak, its z over the disc, the margin to the second peak, and the error of that peak in sigma. Per photo: the query with the highest z, and how many of the 6 queries agree on the place.

**Kill rule (written before the run).** A gate is useful only if, on both data sets and above 800 m, it answers at least 50 % of the photos the pipeline skips with 0 wrong (error > 8 sigma) answers in its answered set. If wrong answers appear at the same threshold on one set and not the other, the gate does not transfer.

Environment: `CC_DATASET`, `CC_STORE`, `CC_LEVELS` (e.g. 3,4,...), `CC_TAG`, `CC_FIRST`, `CC_LIMIT`, `CC_JITTER` (default 5000), `CC_OUT`.

In [1]:
import os, sys, math, csv
import numpy as np, pandas as pd, torch
HEADLESS = os.environ.get("CC_HEADLESS", "../../../../Main_Pipeline_Headless")
sys.path.insert(0, os.path.abspath(HEADLESS))
import settings
settings.LEVELS = [int(v) for v in os.environ.get("CC_LEVELS", "3,4,5,6,7").split(",")]
from geoloc import photo, queries, scoring
from geoloc.cli import read_rows, move_hint
from geoloc.coarse_search import DEVICE, score_map_for_query, pick_peaks
from geoloc.levels import level_for_height
from geoloc.map_store import MapStore

DATASET = os.environ["CC_DATASET"]; STORE = os.environ["CC_STORE"]
TAG = os.environ.get("CC_TAG", "run"); FIRST = int(os.environ.get("CC_FIRST", 0)); LIMIT = int(os.environ.get("CC_LIMIT", 0))
JITTER = float(os.environ.get("CC_JITTER", 5000)); OUT = os.environ.get("CC_OUT", "results/coarse_confidence")
os.makedirs(OUT, exist_ok=True)
store = MapStore(STORE)
rows = read_rows(DATASET)[FIRST:]
if LIMIT: rows = rows[:LIMIT]
print(DEVICE, len(rows), "photos, levels", settings.LEVELS)

cuda 332 photos, levels [3, 4, 5, 6, 7, 8, 9, 10, 11]


In [2]:
def disc_stats(score_map, geometry):
    usable = (score_map > -1) & geometry["inside"]
    values = score_map[usable]
    return float(values.mean()), float(values.std())

records = []
for index, meta in enumerate(rows):
    meta = move_hint(meta, index + FIRST, JITTER, 0) if JITTER > 0 else meta
    camera = photo.read_camera(meta); pose = photo.read_pose(meta)
    prior = store.gps_to_metres(float(meta["lat_sc"]), float(meta["lon_sc"]))
    truth = store.gps_to_metres(float(meta["lat_gtruth"]), float(meta["lon_gtruth"]))
    sigma = scoring.sigma_m(pose["height"], camera["fx"])
    level = level_for_height(pose["height"], camera)
    picture = photo.prepare_photo(os.path.join(DATASET, "images", meta["filename"]), pose, camera, level)
    cache = {}
    for q, query in enumerate(queries.query_list()):
        try:
            canvas, valid = photo.draw_canvas(picture, pose, camera, level, query["heading"], query["scale_factor"])
            score_map, geometry = score_map_for_query(canvas, valid, level, prior, store.features, cache)
        except ValueError:
            continue
        mean, std = disc_stats(score_map, geometry)
        peaks = pick_peaks(score_map, geometry, 5)
        errs = np.hypot(peaks[:, 1] - truth[0], peaks[:, 2] - truth[1]) / sigma
        records.append(dict(photo=index + FIRST, name=meta["filename"], height=pose["height"], level=level, sigma_m=sigma, q=q,
            heading=query["heading"], scale=query["scale_factor"], s1=peaks[0, 0], s2=peaks[1, 0], s3=peaks[2, 0], mean=mean, std=std,
            z1=(peaks[0, 0] - mean) / std, margin=(peaks[0, 0] - peaks[1, 0]) / std, x1=peaks[0, 1], y1=peaks[0, 2], x2=peaks[1, 1], y2=peaks[1, 2],
            err1=errs[0], truth_in_top5=float(errs.min()), true_rank=int(np.argmin(errs)) if errs.min() <= 2 else -1))
    if index % 20 == 0:
        print(index + 1, "/", len(rows), flush=True)
table = pd.DataFrame(records)
table.to_csv(os.path.join(OUT, "queries_%s.csv" % TAG), index=False)
print(len(table), "query rows saved")

1 / 332


21 / 332


41 / 332


61 / 332


81 / 332


101 / 332


... (8 outputs left out)

281 / 332


301 / 332


321 / 332


1992 query rows saved


In [3]:
def points(err):
    return np.where(err <= 1, 10, np.where(err <= 2, 6, np.where(err <= 4, 3, np.where(err <= 8, 0, -20))))

photos = []
for name, group in table.groupby("photo"):
    best = group.loc[group.z1.idxmax()]
    first = group.iloc[0]
    # agreement: queries whose best peak lies within 1 sigma of the best query's peak
    near = np.hypot(group.x1 - best.x1, group.y1 - best.y1) / best.sigma_m <= 1.0
    photos.append(dict(photo=name, height=best.height, sigma_m=best.sigma_m, z=best.z1, margin=best.margin, votes=int(near.sum()), n_q=len(group),
                       err=best.err1, err_plain=first.err1, z_plain=first.z1, top5_hit=float(group.truth_in_top5.min() <= 2)))
photos = pd.DataFrame(photos)
photos["pts"] = points(photos.err.values)
photos.to_csv(os.path.join(OUT, "photos_%s.csv" % TAG), index=False)
bands = [(0, 500), (500, 800), (800, 1000), (1000, 1300), (1300, 1600), (1600, 2200)]
photos["band"] = pd.cut(photos.height, [b[0] for b in bands] + [bands[-1][1]])
print(photos.groupby("band", observed=True).agg(n=("pts", "size"), always_answer_pts=("pts", "mean"), within1=("err", lambda e: (e <= 1).mean()),
      within2=("err", lambda e: (e <= 2).mean()), wrong=("err", lambda e: (e > 8).mean()), z_med=("z", "median"), top5=("top5_hit", "mean")).round(2))

                n  always_answer_pts  within1  within2  wrong  z_med  top5
band                                                                      
(0, 500]      122               7.80     0.54     0.88   0.00  16.84  1.00
(500, 800]     68               7.34     0.46     0.85   0.00  13.70  0.99
(800, 1000]    12               1.33     0.17     0.50   0.17  10.16  0.75
(1000, 1300]   43               4.74     0.28     0.74   0.07   9.91  0.93
(1300, 1600]   45               3.64     0.33     0.64   0.11   7.36  0.80
(1600, 2200]   42               1.43     0.14     0.40   0.14   6.13  0.50


## Does a truth-free confidence separate right from wrong?

AUC of each confidence for "the best coarse peak is within 2 sigma", per band (1.0 = perfect, 0.5 = useless), then the gate table: for each threshold, the share of photos answered, wrong answers (error > 8 sigma), mean points per photo when only gated photos are answered (the rest score 0).

In [4]:
from sklearn.metrics import roc_auc_score
right = (photos.err <= 2).astype(int)
out = []
for band, g in photos.groupby("band", observed=True):
    r = (g.err <= 2).astype(int)
    if r.nunique() < 2: continue
    out.append(dict(band=str(band), n=len(g), right=r.mean().round(2), **{c: round(roc_auc_score(r, g[c]), 2) for c in ["z", "margin", "votes"]}))
print(pd.DataFrame(out).to_string(index=False))

def gate_table(frame, feature, thresholds):
    rows = []
    for t in thresholds:
        sel = frame[frame[feature] >= t]
        rows.append(dict(feature=feature, threshold=t, answered=round(len(sel) / len(frame), 2), wrong=int((sel.err > 8).sum()),
                         right2=round((sel.err <= 2).mean(), 2) if len(sel) else None, pts_per_photo=round(sel.pts.sum() / len(frame), 2)))
    return pd.DataFrame(rows)
high = photos[photos.height >= 800]
print("above 800 m, n =", len(high))
print(gate_table(high, "z", [3, 4, 5, 6, 7, 8, 10]).to_string(index=False))
print(gate_table(high, "votes", [1, 2, 3, 4, 5, 6]).to_string(index=False))
print(gate_table(high, "margin", [0.5, 1, 1.5, 2, 3]).to_string(index=False))

        band   n  right    z  margin  votes
    (0, 500] 122   0.88 0.55    0.53   0.59
  (500, 800]  68   0.85 0.34    0.37   0.53
 (800, 1000]  12   0.50 0.67    0.67   0.67
(1000, 1300]  43   0.74 0.46    0.45   0.66
(1300, 1600]  45   0.64 0.73    0.69   0.62
(1600, 2200]  42   0.40 0.92    0.89   0.69
above 800 m, n = 142
feature  threshold  answered  wrong  right2  pts_per_photo
      z          3      1.00     16    0.59           3.13
      z          4      0.99     15    0.60           3.27
      z          5      0.88      9    0.65           3.83
      z          6      0.68      0    0.74           4.32
      z          7      0.54      0    0.75           3.48
      z          8      0.46      0    0.76           3.04
      z         10      0.27      0    0.72           1.73
feature  threshold  answered  wrong  right2  pts_per_photo
  votes          1      1.00     16    0.59           3.13
  votes          2      0.28      0    0.88           2.04
  votes          3    

In [5]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, feature in zip(axes, ["z", "margin", "votes"]):
    ax.scatter(photos.height, photos[feature], c=np.where(photos.err <= 2, "tab:green", np.where(photos.err <= 8, "tab:orange", "tab:red")), s=14)
    ax.set_xlabel("height (m)"); ax.set_ylabel(feature); ax.set_title(feature + " (green <=2 sigma, orange <=8, red wrong)")
fig.tight_layout(); fig.savefig(os.path.join(OUT, "confidence_%s.png" % TAG), dpi=110)